# M9 · LandLaw AI — Hỏi đáp bằng giọng nói

Chạy trên Google Colab với GPU T4. Demo dùng Hybrid RAG, Qwen 4-bit, nhận dạng tiếng Việt và Edge TTS tùy chọn. Giao diện gồm hội thoại, câu hỏi gợi ý và nguồn trích dẫn.

Chạy các cell lần lượt. Nếu chỉ sửa giao diện, chạy lại cell cuối để giữ model đã nạp. Link Gradio dùng được trong thời gian runtime còn hoạt động; người có link có thể truy cập demo.


In [ ]:
from google.colab import drive
from pathlib import Path
import sys
import os

drive.mount('/content/drive')
# Sửa nếu bạn lưu project ở vị trí khác trên Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa')
assert (PROJECT_ROOT / 'src/landlaw_rag/audio/demo.py').is_file(), f'Không tìm thấy project: {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))


In [ ]:
%pip install -q -r requirements/audio.txt

# Nếu Colab yêu cầu khởi động lại runtime sau cài đặt, khởi động lại rồi chạy lại cell Drive.


In [ ]:
import torch
import gradio as gr

print('Gradio:', gr.__version__)
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))
required_files = ['data/processed/chunks.json', 'indexes/bm25/bm25_index.json.gz']
for relative_path in required_files:
    assert (PROJECT_ROOT / relative_path).is_file(), f'Thiếu dữ liệu: {relative_path}; chạy các milestone xây index trước.'
assert (PROJECT_ROOT / 'indexes/faiss').is_dir(), 'Thiếu indexes/faiss; chạy milestone vector index trước.'


In [ ]:
from landlaw_rag.audio.runtime import load_colab_rag
from landlaw_rag.audio import AudioInteraction, WhisperSTT, EdgeTTS

# Chỉ nạp model một lần; không chạy lại cell này khi chỉ cập nhật UI.
rag = load_colab_rag(PROJECT_ROOT, model_name='Qwen/Qwen2.5-7B-Instruct', strict_citations=False)
interaction = AudioInteraction(
    rag,
    stt=WhisperSTT(
        model_size='vinai/PhoWhisper-medium',
        device='cuda',
        compute_type='float16',
        backend='transformers',
    ),
    tts=EdgeTTS(voice='vi-VN-HoaiMyNeural'),
    output_dir='/content/m9_audio',
)
print('Pipeline đã nạp. Tiếp tục chạy cell mở demo.')


## Mở / cập nhật giao diện

Cell dưới tải lại `demo.py` từ Drive và đóng giao diện cũ trước khi mở bản mới. Không cần nạp lại Qwen khi sửa UI.

Bạn có thể dùng demo ngay trong notebook hoặc mở link `gradio.live` để có không gian rộng hơn và sử dụng microphone. Thu âm → nhận dạng → kiểm tra/sửa câu hỏi → gửi. Checkbox đọc câu trả lời mặc định tắt; bật lên sẽ gửi nội dung trả lời tới Edge TTS trực tuyến. Audio được lưu tạm trong `/content/m9_audio` và mất khi runtime bị xóa.

Lịch sử hiển thị trong phiên; mỗi câu hỏi vào RAG được xử lý độc lập. Muốn đối chiếu, nêu rõ “so sánh Luật Đất đai 2013 và hiện hành” trong câu hỏi.


In [ ]:
import importlib
import landlaw_rag.audio.demo as demo_module

importlib.reload(demo_module)

try:
    demo.close()
except Exception:
    pass

demo = demo_module.build_demo(interaction)
demo.launch(share=True, debug=False)